In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv("data/splits/train.csv")
print(train.shape)

In [ ]:
BEHAVIOR_FEATURES = [
    "acceleration_magnitude",      # aggressiveness
    "steer",                       # steering behavior
    "yaw",                         # stability
    "wheel_slip_magnitude_front",  # traction loss
    "wheel_slip_magnitude_rear",
    "tire_stress_front",           # tire abuse
    "tire_stress_rear",
    "rpm_speed_ratio"              # engine behavior
]
AGG_MAP = {
    "acceleration_magnitude": ["mean", "std"],
    "steer": ["mean", "std"],
    "yaw": ["std"],
    "wheel_slip_magnitude_front": ["mean", "max"],
    "wheel_slip_magnitude_rear": ["mean", "max"],
    "tire_stress_front": ["mean"],
    "tire_stress_rear": ["mean"],
    "rpm_speed_ratio": ["mean", "std"]
}


In [ ]:
lap_features = (train.groupby("lap_number").agg(AGG_MAP))
lap_features.columns = ["_".join(col) for col in lap_features.columns]

print(lap_features.shape)
lap_features.head()

In [ ]:
lap_features.shape


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(lap_features)

In [ ]:
# n_samples = X_scaled.shape[0]

# for k in range(2, min(6, n_samples)):
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     labels = km.fit_predict(X_scaled)
#     score = silhouette_score(X_scaled, labels)
#     print(f"k={k} → silhouette={score:.3f}")

In [ ]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

lap_features["cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
cluster_summary = lap_features.groupby("cluster").mean()
print(cluster_summary)

In [ ]:
CLUSTER_LABELS = {
    0: "Smooth Driving",
    1: "Aggressive Driving"
}

lap_features["driving_style"] = lap_features["cluster"].map(CLUSTER_LABELS)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

KEY_FEATURES = [
    "wheel_slip_magnitude_front_mean",
    "wheel_slip_magnitude_rear_mean",
    "tire_stress_front_mean",
    "tire_stress_rear_mean",
    "steer_std",
    "rpm_speed_ratio_std"
]

cluster_summary[KEY_FEATURES].plot(
    kind="bar",
    figsize=(10,6)
)

plt.title("Driving Behavior Comparison (Cluster Means)")
plt.ylabel("Feature Value")
plt.xlabel("Cluster")
plt.xticks(rotation=0)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7,6))
sns.scatterplot(
    x=X_pca[:,0],
    y=X_pca[:,1],
    hue=lap_features["driving_style"],
    palette=["green", "red"],
    s=100
)

plt.title("Driving Behavior Clusters (Lap-Level PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()


In [ ]:
os.makedirs("artifacts/driving_behavior", exist_ok=True)

joblib.dump(kmeans, "artifacts/driving_behavior/kmeans.pkl")
joblib.dump(scaler, "artifacts/driving_behavior/scaler.pkl")

print("Driving behavior clustering model saved.")

In [ ]:
val = pd.read_csv("data/splits/val.csv")

# Load clustering artifacts
kmeans = joblib.load("artifacts/driving_behavior/kmeans.pkl")
scaler = joblib.load("artifacts/driving_behavior/scaler.pkl")

print(val.shape)
AGG_MAP = {
    "acceleration_magnitude": ["mean", "std"],
    "steer": ["mean", "std"],
    "yaw": ["std"],
    "wheel_slip_magnitude_front": ["mean", "max"],
    "wheel_slip_magnitude_rear": ["mean", "max"],
    "tire_stress_front": ["mean"],
    "tire_stress_rear": ["mean"],
    "rpm_speed_ratio": ["mean", "std"]
}

val_lap_features = (val.groupby("lap_number").agg(AGG_MAP))

val_lap_features.columns = ["_".join(col) for col in val_lap_features.columns]

print(val_lap_features.shape)
val_lap_features.head()
X_val_scaled = scaler.transform(val_lap_features)
val_clusters = kmeans.predict(X_val_scaled)
val_lap_features["cluster"] = val_clusters
CLUSTER_LABELS = {0: "Smooth Driving",1: "Aggressive Driving"}
val_lap_features["driving_style"] = (val_lap_features["cluster"].map(CLUSTER_LABELS))
val_lap_features[["cluster", "driving_style"]]
val_with_style = val.merge(val_lap_features[["driving_style"]].reset_index(),on="lap_number",how="left")

val_with_style[["lap_number", "driving_style"]].head()
print(val_lap_features["driving_style"].value_counts())